<a href="https://colab.research.google.com/github/Ali02-r/Komputasi-Numerik/blob/main/eliminasi_gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eliminasi Gauss untuk SPL 3 Variabel

Notebook ini menyelesaikan tiga persamaan linear dengan tiga bilangan yang belum diketahui (x1, x2, x3) memakai metode **Eliminasi Gauss**. Langkahnya mengikuti dokumen algoritma di repository ini.

**Cara pakai:** jalankan cell dari atas ke bawah dengan `Shift + Enter`. Kalau data soal diubah, jalankan ulang mulai dari Langkah 1.

**Catatan indeks:** pada dokumen algoritma, baris dan kolom dihitung mulai dari 1, sedangkan Python mulai dari 0. Jadi `M[1][1]` di algoritma ditulis `M[0][0]` di kode.

**Alur program secara garis besar:**

| Langkah | Isi | Hasil |
|---------|-----|-------|
| 1 | Data soal (`a` dan `c`) | Koefisien dan konstanta siap dipakai |
| 2 | Susun matriks augmented | Tabel `M` berukuran 3 x 4 |
| 3 | Eliminasi maju | `M` berbentuk tangga, atau berhenti jika tidak ada solusi tunggal |
| 4 | Substitusi mundur | Nilai x3, x2, x1 |
| 5 | Jalankan penyelesaian | Hasil atau pesan "tidak ada solusi tunggal" |
| 6 | Cek jawaban | Bukti bahwa hasilnya cocok dengan soal |
| 7 | Uji kasus khusus | Bukti program menangani soal tanpa solusi tunggal |

Setiap cell kode diberi komentar pada tiap barisnya. Komentar diawali tanda `#` dan tidak mempengaruhi jalannya program.

## Langkah 1: Data soal

Penjelasan cell di bawah ini:

- `a` adalah *list* yang berisi tiga *list* lain. Setiap list di dalamnya mewakili satu persamaan dan berisi koefisien x1, x2, x3 secara berurutan. Misalnya `a[0]` adalah persamaan pertama, dan `a[0][1]` adalah koefisien x2 pada persamaan pertama (nilainya 2).
- `c` adalah list yang berisi konstanta di ruas kanan tiap persamaan, dengan urutan yang sama seperti `a`.
- Untuk soal lain, cukup ganti angka-angkanya. Jika suatu persamaan tidak memuat variabel tertentu, tulis koefisiennya 0.

In [ ]:
# Soal yang diselesaikan (tiga persamaan, tiga bilangan yang belum diketahui):
#   x1 + 2·x2 +   x3 = 2
#  2x1 +   x2 -   x3 = 7
#  3x1        +   x3 = 0

# 'a' adalah list yang berisi 3 list; tiap list di dalamnya adalah satu persamaan.
# Isi tiap list adalah koefisien x1, x2, x3 secara berurutan.
a = [[1, 2,  1],     # persamaan 1: 1·x1 + 2·x2 + 1·x3
     [2, 1, -1],     # persamaan 2: 2·x1 + 1·x2 - 1·x3
     [3, 0,  1]]     # persamaan 3: 3·x1 + 0·x2 + 1·x3 (koefisien x2 sama dengan 0)

# 'c' berisi konstanta di ruas kanan; urutannya sama dengan persamaan di atas.
c = [2, 7, 0]

## Langkah 2: Susun matriks augmented

**Matriks augmented** adalah satu tabel yang menggabungkan koefisien dan konstanta. Dengan satu tabel, setiap operasi pada baris otomatis berlaku untuk koefisien sekaligus konstantanya.

Penjelasan cell di bawah ini:

- `buat_augmented(a, c)` membuat tabel `M` berukuran 3 baris x 4 kolom. Tiga kolom pertama berisi koefisien dan kolom keempat berisi konstanta.
- Setiap angka diubah menjadi bilangan desimal dengan `float(...)` supaya pembagian pada langkah berikutnya tidak dibulatkan.
- `cetak(M)` hanya berfungsi menampilkan tabel dengan rapi dan tidak mengubah isinya.

In [ ]:
def buat_augmented(a, c):
    # Menggabungkan koefisien 'a' dan konstanta 'c' menjadi satu matriks berukuran 3 x 4
    M = []                                    # wadah kosong untuk menampung baris-baris matriks
    for i in range(3):                        # i = 0, 1, 2 -> baris ke-1, ke-2, ke-3
        # float(v) mengubah tiap angka menjadi desimal supaya pembagian tidak membulat.
        # '+ [float(c[i])]' menambahkan konstanta sebagai elemen ke-4 pada baris tersebut.
        baris = [float(v) for v in a[i]] + [float(c[i])]
        M.append(baris)                       # masukkan baris yang sudah jadi ke dalam M
    return M                                  # kembalikan matriks hasilnya


def cetak(M):
    # Hanya untuk menampilkan matriks dengan rapi; isi M tidak berubah
    for baris in M:                           # ambil satu baris dalam satu waktu
        # baris[:3] = tiga elemen pertama (koefisien).
        # f"{v:7.2f}" = angka ditulis selebar 7 karakter dengan 2 angka di belakang koma.
        kiri = "  ".join(f"{v:7.2f}" for v in baris[:3])
        # baris[3] = elemen keempat (konstanta), dipisah garis tegak '|' agar mudah dibaca
        print(f"[ {kiri} | {baris[3]:7.2f} ]")


M = buat_augmented(a, c)                      # bentuk matriks augmented dari data soal
print("Matriks augmented awal:")              # judul sebelum matriks ditampilkan
cetak(M)                                      # tampilkan matriksnya

## Langkah 3: Eliminasi maju

Tujuan langkah ini adalah mengubah `M` menjadi bentuk tangga, yaitu semua elemen di bawah diagonal bernilai 0.

Istilah yang dipakai:

- **Pivot** adalah elemen diagonal `M[k][k]` pada kolom yang sedang diproses. Pivot menjadi pembagi, sehingga tidak boleh 0.
- **Faktor** adalah `M[i][k] / M[k][k]`, yaitu berapa kali baris `k` harus dikurangkan dari baris `i` agar `M[i][k]` menjadi 0.

Contoh pada soal ini, kolom pertama (pivot = 1):

- baris 2: faktor = 2 / 1 = 2, sehingga baris 2 menjadi `[2, 1, -1 | 7] - 2 × [1, 2, 1 | 2] = [0, -3, -3 | 3]`
- baris 3: faktor = 3 / 1 = 3, sehingga baris 3 menjadi `[0, -6, -2 | -6]`

Jika pivot bernilai 0, program mencari baris di bawahnya yang elemen kolomnya tidak 0, lalu menukarnya. Jika tidak ada baris yang bisa ditukar, fungsi mengembalikan `False` karena sistem tidak punya solusi tunggal. Fungsi ini mengubah `M` secara langsung, tanpa membuat salinan.

In [ ]:
def eliminasi_maju(M):
    # Mengubah M menjadi bentuk tangga (segitiga atas).
    # M diubah langsung tanpa membuat salinan, jadi perubahannya terlihat di luar fungsi.
    # Hasil fungsi: True jika berhasil, False jika tidak ada solusi tunggal.
    for k in range(3):                        # k = kolom yang diproses (indeks 0, 1, 2)
        # --- periksa pivot ---
        # Pivot = M[k][k], yaitu elemen diagonal pada kolom k.
        # abs(...) < 1e-12 berarti nilainya dianggap 0 (1e-12 = 0,000000000001, batas toleransi
        # karena bilangan desimal di komputer bisa sedikit meleset dari nilai sebenarnya).
        if abs(M[k][k]) < 1e-12:
            baris_tukar = None                # None = belum ada baris pengganti yang ditemukan
            for i in range(k + 1, 3):         # periksa baris-baris di bawah baris k
                if abs(M[i][k]) > 1e-12:      # jika elemen kolom k pada baris i bukan 0 ...
                    baris_tukar = i           # ... baris i dipilih sebagai pengganti
                    break                     # berhenti mencari, satu baris sudah cukup
            if baris_tukar is None:           # tidak ada baris pengganti sama sekali
                return False                  # tidak ada solusi tunggal, fungsi berhenti di sini
            # Tukar posisi baris k dan baris pengganti (Python bisa menukar dua nilai sekaligus)
            M[k], M[baris_tukar] = M[baris_tukar], M[k]
            # +1 agar nomor baris yang ditampilkan sesuai penomoran biasa (mulai dari 1)
            print(f"Baris {k + 1} ditukar dengan baris {baris_tukar + 1}")

        # --- nolkan elemen di bawah pivot ---
        for i in range(k + 1, 3):             # untuk setiap baris di bawah baris k
            # faktor = berapa kali baris k harus dikurangkan agar M[i][k] menjadi 0
            faktor = M[i][k] / M[k][k]
            for j in range(k, 4):             # proses kolom k sampai kolom terakhir (indeks 3 = konstanta)
                # baris i dikurangi (faktor x baris k), elemen demi elemen
                M[i][j] = M[i][j] - faktor * M[k][j]

        print(f"Setelah kolom {k + 1} diproses:")   # laporan setiap satu kolom selesai
        cetak(M)                              # tampilkan kondisi matriks saat ini

    return True                               # semua kolom selesai, eliminasi berhasil

## Langkah 4: Substitusi mundur

Setelah `M` berbentuk tangga, nilai variabel dicari dari baris paling bawah ke atas:

1. Baris 3 hanya memuat x3, sehingga x3 langsung bisa dihitung.
2. Baris 2 memuat x2 dan x3. Karena x3 sudah diketahui, x2 bisa dihitung.
3. Baris 1 memuat x1, x2, dan x3. Karena x2 dan x3 sudah diketahui, x1 bisa dihitung.

Rumus pada kode didapat dengan memindahkan suku yang sudah diketahui ke ruas kanan, lalu membagi dengan koefisien variabel yang dicari.

In [ ]:
def substitusi_mundur(M):
    # Dipakai setelah M berbentuk tangga. Baris 3 hanya berisi x3, jadi x3 dihitung lebih dulu.
    # M[2][3] = konstanta baris 3, M[2][2] = koefisien x3 pada baris 3
    x3 = M[2][3] / M[2][2]
    # Baris 2: (koefisien x2)·x2 + (koefisien x3)·x3 = konstanta.
    # Suku x3 dipindah ke ruas kanan, lalu dibagi koefisien x2.
    x2 = (M[1][3] - M[1][2] * x3) / M[1][1]
    # Baris 1: caranya sama, tetapi sekarang x2 dan x3 sudah diketahui.
    x1 = (M[0][3] - M[0][1] * x2 - M[0][2] * x3) / M[0][0]
    return x1, x2, x3                         # kembalikan ketiga nilai sekaligus

## Langkah 5: Jalankan penyelesaian

Cell ini menyatukan langkah 3 dan 4:

- `eliminasi_maju(M)` dijalankan lebih dulu. Hasilnya `True` atau `False` disimpan dalam variabel `ada_solusi`.
- Jika `ada_solusi` bernilai `True`, `substitusi_mundur(M)` dijalankan dan hasilnya ditampilkan.
- Jika `False`, program hanya menampilkan pesan bahwa sistem tidak punya solusi tunggal.

Pada `f"x1 = {x1:.4f}"`, bagian `:.4f` berarti nilai ditampilkan dengan 4 angka di belakang koma.

In [ ]:
# eliminasi_maju mengubah M dan mengembalikan True/False; hasilnya disimpan di 'ada_solusi'
ada_solusi = eliminasi_maju(M)

if ada_solusi:                                # True -> lanjut menghitung jawaban
    # fungsi mengembalikan 3 nilai sekaligus, ditampung ke 3 variabel
    x1, x2, x3 = substitusi_mundur(M)
    print()                                   # baris kosong sebagai pemisah
    print("Hasil:")
    print(f"x1 = {x1:.4f}")                   # :.4f = tampilkan 4 angka di belakang koma
    print(f"x2 = {x2:.4f}")
    print(f"x3 = {x3:.4f}")
else:                                         # False -> sistem tidak punya solusi tunggal
    print("Tidak ada solusi tunggal")

## Langkah 6: Cek jawaban

Cara memeriksa jawaban adalah memasukkan x1, x2, x3 ke setiap persamaan awal. Jika ruas kiri sama dengan ruas kanan pada ketiga persamaan, jawabannya benar.

Cell ini memakai data awal `a` dan `c`, bukan `M`, karena `M` sudah berubah selama eliminasi. Pengecekan hanya dijalankan jika `ada_solusi` bernilai `True`, sebab jika tidak, x1, x2, x3 belum pernah dihitung.

In [ ]:
if ada_solusi:                                # cek hanya dilakukan kalau jawabannya ada
    for i in range(3):                        # periksa persamaan 1, 2, 3 satu per satu
        # ruas kiri = koefisien x1, x2, x3 pada persamaan ke-i dikali nilai jawaban, lalu dijumlahkan
        ruas_kiri = a[i][0] * x1 + a[i][1] * x2 + a[i][2] * x3
        # tampilkan ruas kiri berdampingan dengan ruas kanan (c[i]) agar mudah dibandingkan
        print(f"Persamaan {i + 1}: ruas kiri = {ruas_kiri:.4f}, ruas kanan = {c[i]}")

## Langkah 7: Uji soal tanpa solusi tunggal

Cell ini menguji kasus khusus yang sudah dirancang pada algoritma. Persamaan kedua pada soal uji adalah kelipatan dua dari persamaan pertama, sehingga sistem tidak punya solusi tunggal.

Yang seharusnya terjadi:

1. Pada kolom 2, pivot bernilai 0, sehingga program menukar baris 2 dengan baris 3.
2. Pada kolom 3, pivot kembali bernilai 0 dan tidak ada baris di bawahnya untuk ditukar.
3. `eliminasi_maju` mengembalikan `False`, lalu program menampilkan pesan "Tidak ada solusi tunggal".

In [ ]:
# Soal uji: persamaan 2 sengaja dibuat kelipatan dua dari persamaan 1
a_uji = [[1, 2, 1],
         [2, 4, 2],      # = 2 x persamaan 1, sehingga sistem tidak punya solusi tunggal
         [3, 0, 1]]
c_uji = [2, 4, 0]        # konstanta soal uji

M_uji = buat_augmented(a_uji, c_uji)          # bentuk matriks augmented untuk soal uji

# eliminasi_maju mengembalikan True/False, jadi bisa langsung dipakai sebagai syarat if
if eliminasi_maju(M_uji):
    print("Ada solusi tunggal")
else:
    print("Tidak ada solusi tunggal")

## Ringkasan

- Data soal disimpan di `a` (koefisien) dan `c` (konstanta), lalu digabung menjadi matriks augmented `M`.
- `eliminasi_maju` menukar baris bila pivot 0 dan menolkan elemen di bawah pivot, sehingga `M` berbentuk tangga. Jika tidak ada baris yang bisa ditukar, sistem tidak punya solusi tunggal.
- `substitusi_mundur` menghitung x3, lalu x2, lalu x1.
- Jawaban dicek dengan memasukkannya kembali ke persamaan awal.
- Hasil untuk soal contoh: x1 = 1, x2 = 2, x3 = -3.